# Week 1 Day 2: cheap model candidates for the claims agent

This continues the assignment from `labs/week1day2.ipynb`: with a goal of at least 15 / 25 correct, using the LangChain agent, what is the cheapest model you can find?

Three candidates were plugged into the exact same agent as the lab (same `get_policy` and `record_decision` tools, same system prompt), one swap of the `ChatOpenAI` model string at a time:

- **DeepSeek V4 Flash** (`deepseek/deepseek-v4-flash-0731`, low reasoning effort)
- **Qwen3.7 Flash** (`qwen/qwen3.7-flash`, low reasoning effort)
- **Ling-3.0 Flash** (`inclusionai/ling-3.0-flash`, no reasoning effort)

The cells below are just enough setup to reproduce the three runs: imports, the OpenRouter connection, the claims data, and the agent's tools and system prompt, copied from the lab. Then the three runs, then a summary of accuracy, cost and speed against the lab's own Luna baseline.

Note: each run here is a single pass over the 25-claim queue. Scores can swing by a claim or two between runs, so a model should really be run 2-3 times before calling it a consistent winner.

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

In [ ]:
DATA = Path("../../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


claims = load("claims.json")
decisions = load("evals/decisions.json")
labels = {decision["claim_id"]: decision for decision in decisions}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""

recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1

In [ ]:
def run_agent_candidate(model, effort=None):
    """Build the lab's agent on a candidate model and run it over the whole queue."""
    kwargs = {"reasoning_effort": effort} if effort else {}
    candidate_llm = ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5, **kwargs)
    candidate_agent = create_agent(model=candidate_llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

    recorded.clear()
    results = []
    for claim in claims:
        start = time.time()
        result = candidate_agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
        answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
        label = labels[claim["claim_id"]]
        cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
        results.append({
            "model": model,
            "effort": effort,
            "technique": "Agent",
            "claim_id": claim["claim_id"],
            **answer,
            "difficulty": label["difficulty"],
            "right": is_right(answer, label),
            "cost": cost,
            "seconds": round(time.time() - start, 1),
        })

    right = sum(r["right"] for r in results)
    total_cost = sum(r["cost"] for r in results)
    seconds = sum(r["seconds"] for r in results)
    print(f"{model} ({effort}) agent: {right} of {len(results)} right for ${total_cost:.4f} in {seconds:.0f} seconds")
    return results

In [ ]:
# Candidate 1: DeepSeek V4 Flash
deepseek_results = run_agent_candidate("deepseek/deepseek-v4-flash-0731", "low")

In [ ]:
# Candidate 2: Qwen3.7 Flash
qwen_results = run_agent_candidate("qwen/qwen3.7-flash", "low")

In [ ]:
# Candidate 3: Ling-3.0 Flash
ling_results = run_agent_candidate("inclusionai/ling-3.0-flash")

## Summary: cheap model candidates vs Luna (single run each)

| Model | Technique | Effort | Right / 25 | Cost (queue) | Cost / claim | Time (queue) | Time / claim |
|---|---|---|---|---|---|---|---|
| `openai/gpt-6-luna` (lab baseline) | LLM | low | 17 | $0.0042 | $0.00017 | 88s | 3.5s |
| `openai/gpt-6-luna` (lab baseline) | Agent | low | 17 | $0.0087 | $0.00035 | 203s | 8.1s |
| `deepseek/deepseek-v4-flash-0731` | Agent | low | 16 | $0.0097 | $0.00039 | 206s | 8.2s |
| `qwen/qwen3.7-flash` | Agent | low | **18** | $0.0275 | $0.00110 | 1374s (22.9 min) | 55.0s |
| `inclusionai/ling-3.0-flash` | Agent | none | 16 | **$0.0061** | **$0.00024** | 389s (6.5 min) | 15.6s |

**Findings**

- All three candidates clear the assignment's 15/25 bar on this run, but this is one run each, not three. Other community contributions in this repo found scores swing by a claim or two between runs (Ling in particular ranged 14-17 in one write-up), so before claiming a winner, rerun each candidate 2-3 times and report the *worst* score, since "consistently" means the minimum, not the average.
- **Ling-3.0 Flash was the cheapest of the three tested here**, cheaper even than Luna's own agent run ($0.0061 vs $0.0087), while matching DeepSeek's accuracy (16/25). It's the strongest lead for "cheapest model that hits 15/25 consistently," pending confirmation runs.
- **Qwen scored highest (18/25, beating Luna)** but cost over 3x Luna's agent run and took 23 minutes for the queue versus Luna's 3.4 minutes. Its per-token price is low, but it reasons far more verbosely on this task, and the agent runs claims sequentially rather than in parallel, so the extra reasoning tokens turn into a lot of wall-clock time. Price per token isn't price (or time) per claim.
- **DeepSeek landed close to Luna's own numbers** but was slightly behind Luna's agent run on both accuracy and cost in this particular run. OpenRouter's prices shift day to day, so a rerun may land differently.
- None of the three beat Luna's agent run on accuracy; Ling and DeepSeek beat it on cost per claim, Qwen did not.